In [1]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [9]:
from evaluation_helper import get_documents, Questions
from evaluation_utils import llm_structured_retry
from client import client
import json

In [8]:
#Build the insruction of  LLM for document generation
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

### Run a parallel process

In [11]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [12]:
documents = get_documents()

In [13]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
#split the result into separate lists for questions and usage
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [15]:
#Calculate the total token usage for all the requests
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.010775250000000002

In [16]:
#convert to dataframe and save to csv
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [17]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)